# Fracture flag: GPU training

Runs the same `fracture/` code on a Colab GPU with sharper images and more rounds than the CPU run (384 px, 20 rounds).

1. **Runtime > Change runtime type > T4 GPU** (or better), then **Runtime > Run all**.
2. The clone cell asks for a GitHub token (the repo is private). Make one at github.com > Settings > Developer settings > Personal access tokens, with read access to `flatfoot-screener`.
3. At the end, download `fracture_results.zip`.

In [ ]:
!nvidia-smi -L

In [ ]:
import getpass, os
token = getpass.getpass('GitHub token: ')
!git clone -q https://{token}@github.com/swastikjena08-ao/flatfoot-screener.git
del token
%cd flatfoot-screener/ai_models/fracture
!pip -q install imagehash onnx

In [ ]:
# FracAtlas: 338 MB from figshare
!python download.py --out /content/fracatlas

## Settings

In [ ]:
SIZE = 512      # letterbox side (16x16 fracture map)
EPOCHS = 40
BS = 32
W = '/content/work'
os.makedirs(W, exist_ok=True)

In [ ]:
!python prep.py --data /content/fracatlas/FracAtlas --split {W}/split.csv --out {W}/cache.npz --size {SIZE}
!python train.py --cache {W}/cache.npz --init ../../flatfoot_app/model.onnx --out {W}/gpu_run --epochs {EPOCHS} --bs {BS}
!python evaluate.py --cache {W}/cache.npz --model {W}/gpu_run/best.pt --out {W}/gpu_run --overlays {W}/overlays

In [ ]:
import json
r = json.load(open(f'{W}/gpu_run/metrics.json'))
for k in ['test_foot_ankle', 'test_all_leg']:
    print(k, r[k])

In [ ]:
# Download the model and scores
!cd {W} && zip -q -r fracture_results.zip gpu_run/best.pt gpu_run/metrics.json gpu_run/log.json gpu_run/test_predictions.csv overlays
from google.colab import files
files.download(f'{W}/fracture_results.zip')